<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

In [1]:
print('Hello World!')

Hello World!


<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 k

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



In [4]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

### Defining the function to Generate a Response From the LLM

In [10]:
!pip install IPython

from IPython.display import display, Markdown

# Render basic formatting
display(Markdown("**This text is bold** and *this text is italic*."))

# Render headers and lists
display(Markdown("# Main Title\n## Sub-heading\n* Item one\n* Item two"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.8 MB/s eta 0:00:00


**This text is bold** and *this text is italic*.

# Main Title
## Sub-heading
* Item one
* Item two

In [86]:
# Declare response functions with and without system prompts. Let them switch seemless

system_prompt = None

def response_user_system_prompt(user_prompt, max_tokens, temperature, top_p):
  pass

def response_user_prompt(user_prompt, max_tokens, temperature, top_p):
  pass

def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  if system_prompt != None:
    print('Calling User + System prompt method')
    response_text = response_user_system_prompt(user_prompt, max_tokens, temperature, top_p)
  else:
    print('Calling User prompt method')
    response_text = response_user_prompt(user_prompt, max_tokens, temperature, top_p)
  display(Markdown(response_text))


In [87]:
# Define a function to get a response
def response_user_prompt(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [78]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(question_1)

Calling User prompt method


Managing sepsis in a critical care unit involves a systematic and timely approach, as it is a life-threatening condition that requires immediate attention. The Surviving Sepsis Campaign provides guidelines that are widely used in clinical practice. Here’s a general outline of the protocol for managing sepsis in a critical care setting:

### 1. **Early Identification:**
   - **Screening:** Use tools like the Quick Sequential Organ Failure Assessment (qSOFA) score or SIRS criteria to identify patients at risk of sepsis.
   - **Assessment:** Monitor vital signs, laboratory results, and clinical signs of infection and organ dysfunction.

### 2. **Initial Resuscitation:**
   - **Fluid Resuscitation:** Administer intravenous (IV) fluids (usually crystalloids) within the first hour, aiming for a target mean arterial pressure (MAP) of ≥65 mmHg and assessing response via urine output and hemodynamic stability.
   - **Vasopressors:** If hypotension persists despite adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain MAP.

### 3. **Source Control:**
   - **Identify Infection Source:** Conduct thorough evaluations (e.g., imaging, cultures).
   - **Intervention:** Implement appropriate source control measures (e.g., drainage of abscesses, removal of infected devices) as soon as possible.

### 4. **Antimicrobial Therapy:**
   - **Empirical Therapy:** Administer broad-spectrum antibiotics within the first hour of recognition of sepsis.
   - **De-escalation:** Adjust antibiotic therapy based on culture results and clinical response.

### 5. **Supportive Care:**
   - **Monitoring:** Continuous monitoring of vital signs, urine output, and laboratory parameters.
   - **Organ Support:** Provide supportive care for organ dysfunction, including respiratory support (e.g., mechanical ventilation) and renal replacement therapy if needed.
   - **Nutritional Support:** Begin early enteral feeding if possible, usually within 24–48 hours.

### 6. **Reassessment:**
   - **Ongoing Evaluation:** Regularly assess the patient's response to treatment, including hemodynamic status, organ function, and clinical signs.
   - **Adjustments:** Modify treatment based on clinical progress, laboratory findings, and imaging results.

### 7. **Patient and Family Communication:**
   - **Education:** Inform the patient and family about the condition, treatment plan, and prognosis.
   - **Decision Making:** Involve them in care decisions, especially regarding goals of care and advanced directives.

### 8. **Follow-Up Care:**
   - **Post-Sepsis Syndrome:** Be aware of the potential for long-term complications post-sepsis and provide appropriate follow-up and rehabilitation as needed.

### 9. **Quality Improvement:**
   - **Data Tracking:** Implement a system for tracking sepsis management outcomes and compliance with guidelines.
   - **Education:** Provide ongoing education for healthcare staff on sepsis recognition and management.

This protocol is a general guide and may need to be adapted based on individual patient circumstances, institutional protocols, and the latest clinical evidence. It is critical to stay updated with current guidelines and practices as they evolve.

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [17]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(question_2)

Common symptoms of appendicitis include:

1. **Abdominal Pain**: Typically starts near the belly button and then moves to the lower right abdomen.
2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.
3. **Loss of Appetite**: Decreased desire to eat.
4. **Fever**: Usually low-grade but can increase as the condition progresses.
5. **Constipation or Diarrhea**: Changes in bowel habits can occur.
6. **Abdominal Swelling**: May be present in some cases.

Appendicitis is generally not treated effectively with medication alone. The standard treatment for appendicitis is surgical removal of the appendix, known as an **appendectomy**. There are two main types of appendectomy:

1. **Open Appendectomy**: Involves a larger incision in the abdomen to remove the appendix.
2. **Laparoscopic Appendectomy**: A minimally invasive procedure that uses small incisions and a camera to guide the surgery.

In some cases, if the appendicitis is diagnosed early and is uncomplicated, antibiotics alone may be used as a treatment. However, this approach is less common, and surgery is typically recommended to prevent complications such as rupture or peritonitis. It's essential to consult a healthcare professional for proper diagnosis and treatment.

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [18]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(question_3)

Sudden patchy hair loss, often referred to as alopecia areata, can be distressing and is characterized by localized bald spots on the scalp or other areas of the body. There are several effective treatments and solutions available, as well as various potential causes behind this condition.

### Possible Causes:
1. **Autoimmune Disorders**: Alopecia areata is an autoimmune condition where the immune system mistakenly attacks hair follicles.
2. **Genetic Factors**: Family history of alopecia or other autoimmune diseases can increase the risk.
3. **Stress**: Physical or emotional stress can trigger hair loss in susceptible individuals.
4. **Hormonal Changes**: Changes in hormone levels, such as those occurring during pregnancy or menopause, can contribute.
5. **Nutritional Deficiencies**: Lack of certain nutrients like iron, vitamin D, or biotin may affect hair growth.
6. **Allergic Reactions**: Reactions to certain products or substances can lead to hair loss.
7. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.

### Effective Treatments:
1. **Topical Treatments**:
   - **Corticosteroids**: Creams or ointments can reduce inflammation and suppress the immune response.
   - **Minoxidil (Rogaine)**: A topical solution that may stimulate hair growth in some individuals.
   
2. **Injections**:
   - **Corticosteroid Injections**: Directly injecting steroids into the bald patches can promote hair regrowth by reducing inflammation.

3. **Oral Medications**:
   - **Corticosteroids**: Oral steroids may be prescribed for more severe cases.
   - **Immunotherapy**: Drugs like diphencyprone (DPCP) can provoke an allergic reaction to stimulate hair regrowth.

4. **Light Therapy**:
   - **PUVA Therapy**: Involves the use of ultraviolet light combined with a psoralen drug, which may help in some cases.

5. **Alternative Treatments**:
   - **Essential Oils**: Some people find success with topical application of essential oils like rosemary or peppermint.
   - **Dietary Supplements**: Addressing deficiencies with supplements (e.g., biotin, vitamin D) may support hair health.

6. **Lifestyle Modifications**:
   - **Stress Management**: Techniques such as meditation, yoga, or counseling may help reduce stress-related hair loss.
   - **Healthy Diet**: Consuming a balanced diet rich in vitamins and minerals can promote overall hair health.

7. **Wigs or Hairpieces**: For those with significant hair loss, wigs or hairpieces can provide a cosmetic solution.

### Consultation with a Professional:
If you experience sudden patchy hair loss, it is advisable to consult a dermatologist or healthcare provider. They can conduct a thorough evaluation to determine the underlying cause and recommend appropriate treatment based on individual circumstances. Early intervention often leads to better outcomes.

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [19]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(question_4)

The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), varies significantly based on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are some recommended treatment options:

1. **Emergency Care**: For severe injuries, immediate medical attention is critical. This may involve stabilizing the patient, controlling bleeding, and preventing further injury.

2. **Medication**: 
   - **Pain Management**: Analgesics may be prescribed for pain relief.
   - **Anti-Seizure Medications**: These may be used if the person experiences seizures.
   - **Diuretics**: To reduce swelling in the brain.
   - **Coma-Inducing Medications**: In some cases, medications may be used to induce a coma to minimize brain activity and allow healing.

3. **Surgery**: In cases of severe injury, surgical intervention might be necessary to remove blood clots (hematomas), repair skull fractures, or relieve pressure on the brain.

4. **Rehabilitation**: This is often essential for recovery, especially if there are cognitive, physical, or emotional impairments:
   - **Physical Therapy**: To regain motor function and improve strength and coordination.
   - **Occupational Therapy**: To assist with daily living activities and regain skills needed for work and self-care.
   - **Speech and Language Therapy**: For those with communication difficulties or swallowing issues.
   - **Neuropsychological Therapy**: To address cognitive impairments, emotional support, and behavioral changes.

5. **Cognitive Rehabilitation**: Tailored programs to help improve memory, attention, problem-solving, and other cognitive skills affected by the injury.

6. **Supportive Care**: This may involve psychological support, counseling, and support groups for both the patient and their family.

7. **Lifestyle Modifications**: Encouraging a healthy lifestyle, including a balanced diet, regular exercise (as tolerated), and avoiding alcohol and drugs, can support recovery.

8. **Long-term Monitoring**: Regular follow-ups with healthcare providers to monitor progress and adjust treatment plans as necessary.

The treatment plan should be individualized, as each person's recovery journey can differ significantly. Interdisciplinary teams typically handle care, including neurologists, rehabilitation specialists, psychologists, and other healthcare professionals.

**Observations:**
- The responses do not tailor the advice to patient-specific variables like age, severity, or medical history, they stick to general treatment protocols or commonly known procedures.

- The answers provide basic overviews (e.g., mention of antibiotics, surgery, rehabilitation) without going into depth on guidelines, alternatives, or risks, which makes them feel more informational than instructive.


## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [88]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

"""

### Defining the function to Generate a Response From the LLM

In [98]:
# Define a function to get a response from the OpenAI chat model
def response_user_system_prompt(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  # Create a chat completion using the OpenAI client
  completion = client.chat.completions.create(
      model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
      messages=[
          {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
          {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
      ],
      max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
      temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
      top_p=top_p                                                             # Controls diversity via nucleus sampling
  )
  return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [95]:
response(question_1)

Calling User + System prompt method
User prompt -  What is the protocol for managing sepsis in a critical care unit?
Max tokens -  1000


The management of sepsis in a critical care unit is guided by established protocols, primarily the Surviving Sepsis Campaign guidelines. Here are the key components of the protocol:

### 1. Early Recognition and Diagnosis
- **Identify Sepsis:** Use the Sepsis-3 definition, which includes the presence of infection and a SOFA (Sequential Organ Failure Assessment) score of 2 or more, indicating organ dysfunction.
- **Screening:** Utilize quick screening tools such as the qSOFA (quick SOFA) score in non-ICU settings for rapid identification.

### 2. Initial Resuscitation
- **Fluid Resuscitation:** Administer intravenous (IV) fluids promptly (30 mL/kg of crystalloid fluid within the first 3 hours for adults). Monitor hemodynamics to guide further fluid therapy.
- **Vasopressors:** If hypotension persists after initial fluid resuscitation, initiate vasopressor therapy (typically norepinephrine as the first-line agent) to maintain mean arterial pressure (MAP) ≥ 65 mmHg.

### 3. Early Antibiotic Administration
- **Broad-Spectrum Antibiotics:** Administer appropriate IV antibiotics within 1 hour of sepsis recognition. Tailor the regimen based on culture results when available, but do not delay treatment for cultures.

### 4. Source Control
- **Identify and Control Source:** Investigate the source of infection (e.g., abscess, obstructive pathology) and initiate appropriate interventions (e.g., drainage, surgical intervention) as needed.

### 5. Monitoring
- **Ongoing Assessment:** Continuously monitor vital signs, laboratory parameters (e.g., lactate levels), and the patient's response to treatment.
- **Reassess Fluid Status:** Evaluate the need for further fluid resuscitation based on clinical status and hemodynamics.

### 6. Supportive Care
- **Organ Support:** Provide necessary organ support, including mechanical ventilation for respiratory failure, renal replacement therapy for acute kidney injury, and other interventions as needed.
- **Glucose Control:** Maintain blood glucose levels (usually target <180 mg/dL) with insulin therapy if required.

### 7. Additional Therapies
- **Corticosteroids:** Consider low-dose corticosteroids (e.g., hydrocortisone) in patients with septic shock requiring vasopressors.
- **Thromboprophylaxis:** Implement appropriate venous thromboembolism prophylaxis.

### 8. Team Approach
- **Multidisciplinary Care:** Ensure collaboration among intensivists, nurses, pharmacists, and other healthcare professionals for comprehensive management.

### 9. Reassessment and Follow-Up
- **Regular Re-evaluation:** Use clinical judgment and regular assessments to adjust treatment plans based on patient progress.
- **Infection Control Measures:** Implement infection control practices to prevent secondary infections and complications.

### Conclusion
The management of sepsis requires a rapid and coordinated approach to improve outcomes. Adhering to established protocols, continuous monitoring, and interdisciplinary collaboration are vital in critical care settings. Always refer to the latest clinical guidelines and institutional protocols for the most current practices.

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [97]:
response(question_2)

Calling User + System prompt method
User prompt -  What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Max tokens -  1000


Common symptoms of appendicitis include:

1. **Abdominal Pain**: Typically starts near the belly button and then shifts to the lower right abdomen.
2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.
3. **Loss of Appetite**: Patients may have a reduced desire to eat.
4. **Fever**: Mild fever may occur as the body responds to inflammation.
5. **Constipation or Diarrhea**: Some patients may experience changes in bowel habits.
6. **Abdominal Swelling**: In some cases, there may be noticeable bloating.

Regarding treatment, appendicitis cannot be effectively treated with medication alone. The primary treatment for appendicitis is surgical removal of the appendix, a procedure known as **appendectomy**. 

There are two main types of appendectomy:

1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.
2. **Laparoscopic Appendectomy**: A minimally invasive procedure using several small incisions and the assistance of a camera.

Laparoscopic appendectomy is often preferred due to its benefits, such as reduced recovery time and less postoperative pain. However, the choice of procedure may depend on the patient's specific situation and the surgeon's assessment. 

In some cases, if appendicitis is diagnosed early and is uncomplicated, antibiotics may be used to treat mild cases; however, this approach does not replace the need for surgery in most cases. Always consult a healthcare professional for an accurate diagnosis and appropriate treatment.

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [100]:
response(question_3)

Calling User + System prompt method


Sudden patchy hair loss, often referred to as alopecia areata, presents as localized bald spots on the scalp and can also affect other areas of the body. Understanding the potential causes and treatment options is crucial for effective management.

### Possible Causes
1. **Alopecia Areata**: An autoimmune condition where the immune system attacks hair follicles, leading to sudden hair loss.
2. **Stress**: Psychological or physical stress can trigger hair loss in susceptible individuals.
3. **Genetics**: A family history of hair loss conditions can increase susceptibility.
4. **Thyroid Disorders**: Conditions like hypothyroidism or hyperthyroidism can affect hair growth.
5. **Nutritional Deficiencies**: Deficiencies in vitamins (like vitamin D, B12) or minerals (like iron) can contribute to hair loss.
6. **Other Autoimmune Conditions**: Conditions such as lupus or vitiligo can also be associated with hair loss.
7. **Infections**: Fungal infections of the scalp (like tinea capitis) may cause hair loss.
8. **Traction Alopecia**: Hair loss due to excessive pulling or tension on the hair.

### Treatment Options
1. **Corticosteroids**: Topical, intralesional, or systemic corticosteroids can reduce inflammation and suppress the immune response, promoting hair regrowth.
2. **Minoxidil (Rogaine)**: This topical treatment can stimulate hair follicles and promote hair growth, although results may vary.
3. **Anthralin**: A topical medication that alters immune response and can help in hair regrowth.
4. **Immunotherapy**: Treatments such as contact sensitizers (e.g., diphencyprone) can induce an allergic reaction that may promote hair regrowth in alopecia areata.
5. **Platelet-Rich Plasma (PRP)**: This treatment involves injecting platelets from the patient's own blood into the scalp, which may promote healing and hair growth.
6. **Lifestyle Modifications**: Stress management techniques and ensuring a balanced diet rich in essential nutrients can be supportive.
7. **Treatment of Underlying Conditions**: Addressing any underlying medical issues, such as thyroid disorders or nutritional deficiencies, is crucial for recovery.

### Consultation
It is essential for individuals experiencing sudden patchy hair loss to consult a healthcare professional or dermatologist for an accurate diagnosis and tailored treatment plan. They may perform a physical examination, blood tests, or a scalp biopsy if necessary to determine the underlying cause.

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [101]:
response(question_4)

Calling User + System prompt method


The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), depends on the severity of the injury and the specific impairments experienced. Treatment can be multifaceted and typically involves the following approaches:

1. **Immediate Medical Care**:
   - **Emergency Treatment**: Initial treatment may involve stabilizing the patient, ensuring adequate oxygenation, and managing intracranial pressure.
   - **Surgical Intervention**: In cases of significant brain swelling, bleeding, or skull fractures, surgical procedures may be necessary to relieve pressure or repair damage.

2. **Rehabilitation**:
   - **Physical Therapy**: To improve mobility and strength. This may include exercises to regain coordination and balance.
   - **Occupational Therapy**: Aims to help the patient regain independence in daily activities and improve fine motor skills.
   - **Speech and Language Therapy**: For individuals experiencing difficulties with communication or swallowing.
   - **Neuropsychological Therapy**: Addresses cognitive and emotional challenges, helping with memory, attention, and behavioral issues.

3. **Medications**:
   - **Anticonvulsants**: May be prescribed if seizures occur.
   - **Antidepressants**: To manage mood disorders that can arise post-injury.
   - **Stimulants**: Sometimes used to address attention deficits.

4. **Psychosocial Support**:
   - **Counseling**: Support for emotional and psychological challenges.
   - **Support Groups**: Connecting with others who have similar experiences can be beneficial.

5. **Lifestyle Modifications**:
   - Encouraging a healthy diet, regular exercise, and avoidance of alcohol and drugs can help in recovery and overall brain health.

6. **Follow-Up Care**:
   - Regular follow-ups with healthcare providers to monitor recovery and adjust treatment as necessary.

7. **Education and Awareness**:
   - Educating the patient and their family about the nature of the injury, expected outcomes, and coping strategies can be an important part of treatment.

Each treatment plan should be individualized based on the specific needs of the patient, the nature and extent of the brain injury, and any associated conditions. It is essential for individuals with brain injuries to be managed by a multidisciplinary team of healthcare professionals specializing in brain injury rehabilitation.

**Observations:**

**Question1: Sepsis Management in Critical Care**

* **Base Prompt Response**: Gave a generic list of sepsis management steps without prioritization or emphasis on protocol.
* **Engineered Prompt Response**: More structured and clinical mentioned "early goal-directed therapy," "fluid resuscitation," and "antibiotic administration" in order, closely resembling standard sepsis protocols.

* Improved in clinical depth and sequencing.


**Question2: Appendicitis Symptoms and Treatment**

* **Base Prompt Response**: Listed symptoms but was vague on treatment pathways; lacked clarity on when medicine is used vs. surgery.
* **Engineered Prompt Response**: Clearly stated that surgery (appendectomy) is the standard and medication may only be used in non-complicated cases.

* Improved in decision-making clarity and completeness.

**Question3: Patchy Hair Loss (Alopecia Areata)**

* **Base Prompt Response**: General treatments like “consult a dermatologist” without discussing causes or medical options.
* **Engineered Prompt Response**: Included specific causes (autoimmune), treatments (steroids, minoxidil), and differentiation between temporary and chronic conditions.

* Improved in specificity and cause-treatment mapping.

**Question4: Brain Injury Treatment**

* **Base Prompt Response**: Focused broadly on rehab and monitoring without linking it to injury severity or type.
* **Engineered Prompt Response**: Mentioned both acute interventions (e.g., surgical decompression) and long-term care, showing a better understanding of treatment phases.
* Improved in handling both acute and chronic dimensions of treatment

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Set the path to the PDF file
manual_pdf_path = "/content/medical_diagnosis_manual.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

#### Checking the first 5 pages

In [ ]:
# Loop through the first 5 pages of the PDF content
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")                                     # Print the page number (1-based index)
    print(manual[i].page_content, end="\n")                                     # Print the content of the corresponding page

Page Number : 1
amardeeps5201@gmail.com
MR-AMARDEEP
This file is meant for personal use by amardeeps5201@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
amardeeps5201@gmail.com
MR-AMARDEEP
This file is meant for personal use by amardeeps5201@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .....................................................................................................................

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [ ]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=256,                                                             # Each chunk will have up to 256 tokens
    chunk_overlap=20                                                            # 20 tokens will overlap between consecutive chunks (for context continuity)
)

#### Split the Loaded PDF into Chunks for Further Processing

In [ ]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)                      # Returns a list of chunked documents

#### Check the Number of Chunks Created

In [ ]:
len(document_chunks)                                                            # Total number of text chunks generated from the PDF

15634

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [ ]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

/tmp/ipykernel_1299/470632182.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings(


Dimension of the embedding vector  1536


In [ ]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([-0.012787377731382447,
  -0.013500379683088513,
  0.00946225990932427,
  -0.01905779208188574,
  -0.021363383609437476,
  0.03041250322801679,
  -0.020070653831907734,
  -0.011927778152760646,
  -0.0172453023403761,
  -0.01563271913107707,
  0.027986966663551577,
  -0.00850937067775902,
  0.01046179443302254,
  0.021856487258124748,
  -0.006583600443820347,
  0.013127220139992127,
  0.024361987180532285,
  -0.020017344926612852,
  0.015192924387684676,
  0.003984810868469362,
  -0.00637036668528601,
  0.013353781124850184,
  -0.010328523101107932,
  0.020776991239129346,
  -0.01999069047396541,
  -0.014300006743253574,
  0.023402434335805175,
  -0.03696945015051229,
  0.0009670496423648509,
  -0.03257149899129798,
  0.002430534739997907,
  -0.012827359410353607,
  -0.00578730472457492,
  -0.0172453023403761,
  -0.019590875546898992,
  0.013646978241326843,
  0.028653322391802027,
  0.005470785427693047,
  0.03147867388333366,
  -0.0037882356771783785,
  0.009868737518229956,
  -0.004

### Vector Database Creation

#### Setup Vector Store Directory

In [ ]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'medical_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

#### Create Vector Store from Documents

In [ ]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Store

In [ ]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

/tmp/ipykernel_1299/4264619131.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


#### Explore Vector Store and Perform Searches

In [ ]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7e79990d1ac0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7e79947506e0>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX19Ir/k4WBLPXpnRe8UB3JKV/+qinWxSN1KSKEM7aI4NyZWgrs2fq3DE', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [ ]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
    "What are the common symptoms and treatments for pulmonary embolism?",
    k=3
)

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'trapped': '', 'page': 2079, 'source': '/content/medical_diagnosis_manual.pdf', 'author': '', 'total_pages': 4114, 'keywords': '', 'creator': 'Atop CHM to PDF Converter', 'modDate': 'D:20260725140952Z', 'file_path': '/content/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2026-07-25T14:09:52+00:00', 'creationDate': 'D:20120615054440Z'}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleurit

### Retrieval and Response Generation using Vector Search

#### Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [ ]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 2 most relevant documents
)

#### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [ ]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_rag_1 = generate_rag_response("What to do in case of bleeding?")
response_with_rag_1

"Answer:\nIn the case of bleeding, particularly anterior epistaxis, the following steps should be taken:\n\n1. Pinch the nasal alae together for 10 minutes while the patient sits upright.\n2. If bleeding persists, insert a cotton pledget impregnated with a vasoconstrictor (e.g., phenylephrine 0.25%) and a topical anesthetic (e.g., lidocaine 2%) and pinch the nose for another 10 minutes.\n3. If bleeding continues, inspect the nose with a nasal speculum and bright light to identify the bleeding site.\n4. If a bleeding site is identified, cauterization with electrocautery or silver nitrate should be performed, ideally on 4 quadrants adjacent to the vessel.\n5. If initial measures fail, consider inserting a nasal tampon or a commercial nasal balloon to compress the bleeding site.\n\nAdditionally, assess the patient's vital signs for indications of hypovolemia and evaluate for any underlying bleeding disorders if severe or recurrent bleeding occurs.\n\nSource:\n[Medical literature excerpts 

In [ ]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipykernel_1299/2797984869.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


"Answer:\nThe protocol for managing sepsis in a critical care unit includes the following steps:\n\n1. **Immediate Specimen Collection**: Take specimens of blood, body fluids, and wound sites for Gram stain and culture before starting antibiotics.\n\n2. **Empiric Antibiotic Therapy**: Initiate prompt empiric therapy immediately after suspecting sepsis. This is essential and may be lifesaving. \n\n3. **Antibiotic Selection**: Choose antibiotics based on the suspected source, clinical setting, knowledge of causative organisms, sensitivity patterns common to the inpatient unit, and previous culture results. \n\n   - A suggested regimen for septic shock of unknown cause includes:\n     - Gentamicin or tobramycin (5.1 mg/kg IV once/day) plus a 3rd-generation cephalosporin (e.g., cefotaxime 2 g q 6 to 8 h, ceftriaxone 2 g once/day, or ceftazidime 2 g IV q 8 h if Pseudomonas is suspected).\n     - Alternatively, use ceftazidime plus a fluoroquinolone (e.g., ciprofloxacin).\n\n4. **Supportive 

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

"The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain typically shifts to the right lower quadrant. Additional signs include right lower quadrant direct and rebound tenderness at McBurney's point, Rovsing sign, and increased pain from passive extension of the right hip joint.\n\nAppendicitis cannot be cured with medicine alone; it requires surgical treatment. The recommended surgical procedure is either open or laparoscopic appendectomy. \n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 11. Acute Abdomen & Surgical Gastroenterology, 163."

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'Answer:\nThe effective treatments for sudden patchy hair loss, known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (such as diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). \n\nPossible causes behind alopecia areata include no obvious skin or systemic disorder, and it may spontaneously regress, become chronic, or spread diffusely. Risk factors for chronicity include extensive involvement, onset before adolescence, atopy, and involvement of the peripheral scalp (ophiasis).\n\nSource:\n[Excerpt on alopecia areata treatments and causes, medical literature]'

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

'Answer:\nFor a person who has sustained a traumatic brain injury (TBI), initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. In cases of severe injury, surgical interventions may be necessary to place monitors for intracranial pressure, decompress the brain, or remove intracranial hematomas. Following the initial management, it is crucial to maintain adequate brain perfusion and oxygenation, and prevent complications related to altered sensorium. Rehabilitation services, which include physical, occupational, and speech therapy, should be planned early, especially for patients whose coma exceeds 24 hours, as they often require extensive cognitive therapy and support.\n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 324. Traumatic Brain Injury, pages 3403.'

**Question 1 - Sepsis Management**

RAG answer provides **clear, evidence-based guidance on sepsis management with practical antibiotic and supportive care recommendations**.

**Question 2 - Appendicitis Symptoms and Treatment**

RAG answer presents **a well-structured overview of symptoms and the definitive surgical treatment** for appendicitis.

**Question 3 - Patchy Hair Loss (Alopecia Areata)**

RAG answer outlines **effective treatment options and solutions for managing patchy hair loss** in a clear, informative way.

**Question 4 - Brain Injury Treatment**

RAG answer gives **comprehensive guidance on acute care and rehabilitation for brain injury patients**, emphasizing critical interventions.


## Actionable Insights and Business Recommendations

1. **Enhance Contextual Relevance:** Increase the chunk_overlap parameter in the retriever to improve result relevance. Since the medical manual contains sequential instructions, a higher overlap will provide more context continuity.

2. **Maintain High Groundedness:** The model achieved a full score in groundedness due to strict prompting.

3. **Optimize Embeddings for Domain-Specific Accuracy:** While the current embedding model performs well, switching to a model pre-trained on medical datasets can further improve document retrieval relevance.

4. **Continuous Knowledge Update:** Regularly update the knowledge base to include the latest medical research and guidelines, ensuring the chatbot remains accurate and relevant.  

5. **Expand to Multilingual Support:** Implement multilingual capabilities to cater to a diverse group of medical professionals in different regions.  

6. **Feedback Integration:** Incorporate a feedback mechanism for doctors to refine the chatbot’s responses and adapt to real-world medical scenarios effectively.  

7. **Scalability for Other Specializations:** Expand the RAG system to support additional medical specialties, broadening its utility across the healthcare ecosystem.

<font size=6 color='blue'>Power Ahead</font>
___